# Experiment 5 :: Subword Tokenization and POS Tagging

- 5.1 Subword tokenization with BPE (a. pretrained, b. from scratch)
- 5.2 Subword tokenization with SentencePiece (a. pretrained, b. from scratch)
- 5.3 POS tagging of a sentence (a. spaCy, b. NLTK)
- 5.4 POS tagging with token frequency (spaCy)

In [ ]:
!pip install -q transformers sentencepiece spacy nltk
!python -m spacy download en_core_web_sm

In [2]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('tagsets')
nltk.download('tagsets_json')

[nltk_data] Downloading package punkt to /Users/apple/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/apple/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/apple/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/apple/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets to /Users/apple/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package tagsets_json to
[nltk_data]     /Users/apple/nltk_data...
[nltk_data]   Package tagsets_json is already up-to-date!


True

## Input data

In [3]:
import re

text = open('input_sub_word_data.txt', encoding='utf-8').read()
flat = ' '.join(text.split())
sentences = [s.strip() for s in re.findall(r'[^.]+\.', flat)]

sentence = sentences[0]
rare_sentence = [s for s in sentences if 'hydrokinetic' in s][0]

print('Characters:', len(text))
print('Sentences:', len(sentences))
print('Words:', len(flat.split()))
print()
print('Sentence:', sentence)
print()
print('Rare word sentence:', rare_sentence)

Characters: 6682
Sentences: 66
Words: 947

Sentence: Natural language processing is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language.

Rare word sentence: The tokenizer can then be tested on new words such as naturalization, internationalization, computationally, unbelievable, preprocessing, classification, communication, renewable, hydrokinetic, multilingual, understanding, and modernization.


In [4]:
def show(tokens, ids):
    print(f"{'#':<5}{'TOKEN':<24}{'TOKEN ID'}")
    for i, (token, token_id) in enumerate(zip(tokens, ids), 1):
        print(f'{i:<5}{token:<24}{token_id}')
    print('Total tokens:', len(tokens))

## 5.1 Subword tokenization using a BPE tokenizer

### 5.1 a. Using a pretrained model

In [5]:
from transformers import AutoTokenizer

bpe = AutoTokenizer.from_pretrained('gpt2')

print('Vocabulary size:', bpe.vocab_size)

/private/tmp/claude-501/-Users-apple-Documents-GitHub-coding-ninjas-landing/6cab71ae-eb3e-4a73-8a3f-54b1185767fd/scratchpad/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/private/tmp/claude-501/-Users-apple-Documents-GitHub-coding-ninjas-landing/6cab71ae-eb3e-4a73-8a3f-54b1185767fd/scratchpad/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Vocabulary size: 50257


In [6]:
tokens = bpe.tokenize(sentence)
ids = bpe.convert_tokens_to_ids(tokens)

print(sentence)
print()
show(tokens, ids)

Natural language processing is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language.

#    TOKEN                   TOKEN ID
1    Natural                 35364
2    Ġlanguage               3303
3    Ġprocessing             7587
4    Ġis                     318
5    Ġa                      257
6    Ġbranch                 8478
7    Ġof                     286
8    Ġartificial             11666
9    Ġintelligence           4430
10   Ġthat                   326
11   Ġenables                13536
12   Ġcomputers              9061
13   Ġto                     284
14   Ġunderstand             1833
15   ,                       11
16   Ġinterpret              6179
17   ,                       11
18   Ġand                    290
19   Ġgenerate               7716
20   Ġhuman                  1692
21   Ġlanguage               3303
22   .                       13
Total tokens: 22


In [7]:
tokens = bpe.tokenize(rare_sentence)
ids = bpe.convert_tokens_to_ids(tokens)

print(rare_sentence)
print()
show(tokens, ids)

The tokenizer can then be tested on new words such as naturalization, internationalization, computationally, unbelievable, preprocessing, classification, communication, renewable, hydrokinetic, multilingual, understanding, and modernization.

#    TOKEN                   TOKEN ID
1    The                     464
2    Ġtoken                  11241
3    izer                    7509
4    Ġcan                    460
5    Ġthen                   788
6    Ġbe                     307
7    Ġtested                 6789
8    Ġon                     319
9    Ġnew                    649
10   Ġwords                  2456
11   Ġsuch                   884
12   Ġas                     355
13   Ġnatural                3288
14   ization                 1634
15   ,                       11
16   Ġinternational          3230
17   ization                 1634
18   ,                       11
19   Ġcomput                 2653
20   ationally               15208
21   ,                       11
22   Ġunbelievabl

### 5.1 b. Without using a pretrained model

In [8]:
from collections import Counter

word_freqs = Counter(re.findall(r'[a-z]+', text.lower()))

print('Unique words in corpus:', len(word_freqs))
print(word_freqs.most_common(10))

Unique words in corpus: 365
[('the', 48), ('a', 28), ('and', 27), ('can', 23), ('of', 22), ('subword', 17), ('to', 15), ('be', 15), ('is', 14), ('word', 13)]


In [9]:
def learn_bpe(word_freqs, num_merges):
    vocab = {tuple(list(word) + ['</w>']): freq for word, freq in word_freqs.items()}
    merges = []

    for _ in range(num_merges):
        pairs = Counter()
        for symbols, freq in vocab.items():
            for i in range(len(symbols) - 1):
                pairs[symbols[i], symbols[i + 1]] += freq

        if not pairs:
            break

        best = max(pairs, key=pairs.get)
        merges.append(best)

        merged_vocab = {}
        for symbols, freq in vocab.items():
            merged, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best:
                    merged.append(symbols[i] + symbols[i + 1])
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            merged = tuple(merged)
            merged_vocab[merged] = merged_vocab.get(merged, 0) + freq
        vocab = merged_vocab

    return merges, vocab


NUM_MERGES = 300
merges, learned = learn_bpe(word_freqs, NUM_MERGES)
ranks = {pair: i for i, pair in enumerate(merges)}

print('Merges learned:', len(merges))
print()
print(f"{'#':<5}{'MERGED PAIR':<20}{'NEW SYMBOL'}")
for i, (a, b) in enumerate(merges[:25], 1):
    print(f'{i:<5}{a + " + " + b:<20}{a + b}')

Merges learned: 300

#    MERGED PAIR         NEW SYMBOL
1    e + </w>            e</w>
2    s + </w>            s</w>
3    e + n               en
4    i + n               in
5    d + </w>            d</w>
6    e + r               er
7    o + r               or
8    a + n               an
9    t + h               th
10   o + n               on
11   a + t               at
12   a + r               ar
13   r + e               re
14   a + l               al
15   in + g              ing
16   t + o               to
17   y + </w>            y</w>
18   ing + </w>          ing</w>
19   en + t              ent
20   i + on              ion
21   th + e</w>          the</w>
22   w + or              wor
23   l + e               le
24   at + ion            ation
25   er + </w>           er</w>


In [10]:
subwords = {symbol for symbols in learned for symbol in symbols}
subwords |= {char for word in word_freqs for char in word}
subwords.add('</w>')

vocabulary = {'<unk>': 0}
for symbol in sorted(subwords):
    vocabulary[symbol] = len(vocabulary)

print('Vocabulary size:', len(vocabulary))
print(list(vocabulary.items())[:20])

Vocabulary size: 278
[('<unk>', 0), ('</w>', 1), ('a', 2), ('a</w>', 3), ('ab', 4), ('able</w>', 5), ('act', 6), ('ain', 7), ('aining</w>', 8), ('al', 9), ('al</w>', 10), ('algorith', 11), ('algorithm</w>', 12), ('alization</w>', 13), ('all', 14), ('ally</w>', 15), ('an', 16), ('an</w>', 17), ('and</w>', 18), ('ang', 19)]


In [11]:
def encode_word(word, table):
    symbols = list(word) + ['</w>']

    while len(symbols) > 1:
        candidates = [(table[symbols[i], symbols[i + 1]], i)
                      for i in range(len(symbols) - 1)
                      if (symbols[i], symbols[i + 1]) in table]
        if not candidates:
            break
        _, i = min(candidates)
        symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]

    return symbols


def encode(sent):
    tokens = [token for word in re.findall(r'[a-z]+', sent.lower()) for token in encode_word(word, ranks)]
    return tokens, [vocabulary.get(token, 0) for token in tokens]

In [12]:
tokens, ids = encode(sentence)

print(sentence)
print()
show(tokens, ids)

Natural language processing is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language.

#    TOKEN                   TOKEN ID
1    natural</w>             172
2    language</w>            145
3    processing</w>          198
4    is</w>                  133
5    a</w>                   3
6    b                       32
7    r                       203
8    an                      16
9    ch</w>                  40
10   of</w>                  179
11   art                     24
12   ific                    119
13   i                       113
14   al</w>                  10
15   in                      123
16   t                       229
17   el                      72
18   li                      156
19   g                       105
20   ence</w>                77
21   that</w>                234
22   en                      74
23   ab                      4
24   les</w>                 154
25   compu                   46
26  

In [13]:
tokens, ids = encode(rare_sentence)

print(rare_sentence)
print()
show(tokens, ids)

The tokenizer can then be tested on new words such as naturalization, internationalization, computationally, unbelievable, preprocessing, classification, communication, renewable, hydrokinetic, multilingual, understanding, and modernization.

#    TOKEN                   TOKEN ID
1    the</w>                 235
2    tokenizer</w>           242
3    can</w>                 38
4    th                      233
5    en</w>                  75
6    be</w>                  34
7    t                       229
8    es                      85
9    t                       229
10   ed</w>                  71
11   on</w>                  182
12   ne                      173
13   w                       267
14   </w>                    1
15   words</w>               273
16   such</w>                226
17   as</w>                  26
18   nat                     171
19   ur                      257
20   alization</w>           13
21   internation             128
22   alization</w>           13
23 

In [14]:
print(f"{'WORD':<26}{'SUBWORD UNITS'}")
for word in ['naturalization', 'internationalization', 'computationally', 'unbelievable',
             'preprocessing', 'classification', 'communication', 'renewable',
             'hydrokinetic', 'multilingual', 'understanding', 'modernization']:
    print(f'{word:<26}{" ".join(encode_word(word, ranks))}')

WORD                      SUBWORD UNITS
naturalization            nat ur alization</w>
internationalization      internation alization</w>
computationally           comput ation ally</w>
unbelievable              un b el i ev able</w>
preprocessing             pre processing</w>
classification            cl ass ific ation</w>
communication             com m un ic ation</w>
renewable                 r en e w able</w>
hydrokinetic              h y d r o k in e t ic </w>
multilingual              m ult il ingu al</w>
understanding             underst an ding</w>
modernization             mod er n ization</w>


In [15]:
for num_merges in [50, 100, 200, 300, 500]:
    trial_merges, _ = learn_bpe(word_freqs, num_merges)
    trial_ranks = {pair: i for i, pair in enumerate(trial_merges)}
    counts = [len(encode_word(word, trial_ranks)) for word in word_freqs]
    print(f'merges={num_merges:<6}tokens per word={sum(counts) / len(counts):.2f}')

merges=50    tokens per word=5.30
merges=100   tokens per word=4.50


merges=200   tokens per word=3.67


merges=300   tokens per word=3.11


merges=500   tokens per word=2.31


## 5.2 Subword tokenization using a SentencePiece tokenizer

### 5.2 a. Using a pretrained model

In [16]:
sp_pretrained = AutoTokenizer.from_pretrained('albert-base-v2')

print('Tokenizer:', type(sp_pretrained).__name__)
print('Vocabulary size:', sp_pretrained.vocab_size)

Tokenizer: AlbertTokenizerFast
Vocabulary size: 30000


In [17]:
tokens = sp_pretrained.tokenize(sentence)
ids = sp_pretrained.convert_tokens_to_ids(tokens)

print(sentence)
print()
show(tokens, ids)

Natural language processing is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language.

#    TOKEN                   TOKEN ID
1    ▁natural                1112
2    ▁language               816
3    ▁processing             5511
4    ▁is                     25
5    ▁a                      21
6    ▁branch                 1686
7    ▁of                     16
8    ▁artificial             6809
9    ▁intelligence           2872
10   ▁that                   30
11   ▁enables                14645
12   ▁computers              7774
13   ▁to                     20
14   ▁understand             1369
15   ,                       15
16   ▁interpret              11584
17   ,                       15
18   ▁and                    17
19   ▁generate               7920
20   ▁human                  585
21   ▁language               816
22   .                       9
Total tokens: 22


In [18]:
tokens = sp_pretrained.tokenize(rare_sentence)
ids = sp_pretrained.convert_tokens_to_ids(tokens)

print(rare_sentence)
print()
show(tokens, ids)

The tokenizer can then be tested on new words such as naturalization, internationalization, computationally, unbelievable, preprocessing, classification, communication, renewable, hydrokinetic, multilingual, understanding, and modernization.

#    TOKEN                   TOKEN ID
1    ▁the                    14
2    ▁to                     20
3    ken                     2853
4    izer                    11907
5    ▁can                    92
6    ▁then                   94
7    ▁be                     44
8    ▁tested                 7631
9    ▁on                     27
10   ▁new                    78
11   ▁words                  715
12   ▁such                   145
13   ▁as                     28
14   ▁natural                1112
15   ization                 1829
16   ,                       15
17   ▁international          294
18   ization                 1829
19   ,                       15
20   ▁computational          16439
21   ly                      102
22   ,                     

### 5.2 b. Without using a pretrained model

In [19]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input='input_sub_word_data.txt',
    model_prefix='sp_corpus',
    vocab_size=500,
    model_type='bpe',
    character_coverage=1.0,
    minloglevel=2,
)

sp = spm.SentencePieceProcessor(model_file='sp_corpus.model')

print('Vocabulary size:', sp.get_piece_size())
print([sp.id_to_piece(i) for i in range(20)])

Vocabulary size: 500
['<unk>', '<s>', '</s>', '▁t', 'en', 'in', 'er', '▁a', 'or', '▁c', 'on', 'es', 'at', '▁s', 'ar', 'an', 'ing', '▁th', 're', 'ent']


In [20]:
tokens = sp.encode(sentence, out_type=str)
ids = sp.encode(sentence)

print(sentence)
print()
show(tokens, ids)

Natural language processing is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language.

#    TOKEN                   TOKEN ID
1    ▁N                      213
2    atural                  174
3    ▁language               118
4    ▁processing             178
5    ▁is                     84
6    ▁a                      7
7    ▁b                      32
8    r                       459
9    an                      15
10   ch                      210
11   ▁of                     58
12   ▁ar                     399
13   t                       458
14   ific                    327
15   ial                     384
16   ▁int                    156
17   el                      61
18   l                       463
19   ig                      298
20   ence                    153
21   ▁that                   273
22   ▁en                     321
23   ab                      73
24   les                     314
25   ▁compu                  278

In [21]:
tokens = sp.encode(rare_sentence, out_type=str)
ids = sp.encode(rare_sentence)

print(rare_sentence)
print()
show(tokens, ids)

The tokenizer can then be tested on new words such as naturalization, internationalization, computationally, unbelievable, preprocessing, classification, communication, renewable, hydrokinetic, multilingual, understanding, and modernization.

#    TOKEN                   TOKEN ID
1    ▁The                    139
2    ▁tokenizer              120
3    ▁can                    52
4    ▁then                   340
5    ▁be                     41
6    ▁t                      3
7    es                      11
8    ted                     186
9    ▁on                     189
10   ▁new                    437
11   ▁words                  117
12   ▁such                   272
13   ▁as                     83
14   ▁natural                233
15   ization                 160
16   ,                       473
17   ▁international          294
18   ization                 160
19   ,                       473
20   ▁compu                  278
21   t                       458
22   ational                 176

In [22]:
print(f"{'WORD':<26}{'SUBWORD UNITS'}")
for word in ['naturalization', 'internationalization', 'computationally', 'unbelievable',
             'preprocessing', 'classification', 'communication', 'renewable',
             'hydrokinetic', 'multilingual', 'understanding', 'modernization']:
    print(f'{word:<26}{" ".join(sp.encode(word, out_type=str))}')

WORD                      SUBWORD UNITS
naturalization            ▁natural ization
internationalization      ▁international ization
computationally           ▁compu t ational ly
unbelievable              ▁un b el i e v able
preprocessing             ▁preprocessing
classification            ▁classification
communication             ▁com mun ication
renewable                 ▁r en e w able
hydrokinetic              ▁h y d ro k ine tic
multilingual              ▁m ult iling ual
understanding             ▁understand ing
modernization             ▁mod ern ization


## 5.3 Part-of-Speech tagging

In [23]:
pos_text = 'The young student is reading an interesting book in the library.'

print(pos_text)

The young student is reading an interesting book in the library.


### 5.3 a. Using spaCy

In [24]:
import spacy

nlp = spacy.load('en_core_web_sm')
doc = nlp(pos_text)

unique = {}
for token in doc:
    unique.setdefault((token.text.lower(), token.pos_, token.tag_), token.text)

print(f"{'TOKEN':<14}{'POS':<8}{'TAG':<8}{'DESCRIPTION'}")
for (_, pos, tag), original in unique.items():
    print(f'{original:<14}{pos:<8}{tag:<8}{spacy.explain(tag)}')

TOKEN         POS     TAG     DESCRIPTION
The           DET     DT      determiner
young         ADJ     JJ      adjective (English), other noun-modifier (Chinese)
student       NOUN    NN      noun, singular or mass
is            AUX     VBZ     verb, 3rd person singular present
reading       VERB    VBG     verb, gerund or present participle
an            DET     DT      determiner
interesting   ADJ     JJ      adjective (English), other noun-modifier (Chinese)
book          NOUN    NN      noun, singular or mass
in            ADP     IN      conjunction, subordinating or preposition
library       NOUN    NN      noun, singular or mass
.             PUNCT   .       punctuation mark, sentence closer


### 5.3 b. Using NLTK

In [25]:
from nltk.tokenize import word_tokenize


def penn_descriptions():
    for path in ['help/tagsets/upenn_tagset.pickle', 'help/tagsets_json/PY3/upenn_tagset.json']:
        try:
            return {tag: value[0] for tag, value in nltk.data.load(path).items()}
        except Exception:
            continue
    return {}


descriptions = penn_descriptions()
tagged = nltk.pos_tag(word_tokenize(pos_text))

unique = {}
for token, tag in tagged:
    unique.setdefault((token.lower(), tag), token)

print(f"{'TOKEN':<14}{'TAG':<8}{'DESCRIPTION'}")
for (_, tag), original in unique.items():
    print(f'{original:<14}{tag:<8}{descriptions.get(tag, "")}')

TOKEN         TAG     DESCRIPTION
The           DT      determiner
young         JJ      adjective or numeral, ordinal
student       NN      noun, common, singular or mass
is            VBZ     verb, present tense, 3rd person singular
reading       VBG     verb, present participle or gerund
an            DT      determiner
interesting   JJ      adjective or numeral, ordinal
book          NN      noun, common, singular or mass
in            IN      preposition or conjunction, subordinating
library       NN      noun, common, singular or mass
.             .       sentence terminator


/private/tmp/claude-501/-Users-apple-Documents-GitHub-coding-ninjas-landing/6cab71ae-eb3e-4a73-8a3f-54b1185767fd/scratchpad/venv/lib/python3.9/site-packages/nltk/app/__init__.py:45: UserWarning: nltk.app.wordfreq not loaded (requires the matplotlib library).
  warnings.warn("nltk.app.wordfreq not loaded (requires the matplotlib library).")


## 5.4 POS tagging with frequency using spaCy

In [26]:
doc = nlp(flat)

counts = Counter((token.text.lower(), token.pos_, token.tag_) for token in doc if not token.is_space)
rows = sorted(counts.items(), key=lambda item: (-item[1], item[0][0]))

print('Total tokens:', sum(counts.values()))
print('Unique tokens:', len(counts))
print()
print(f"{'TOKEN':<22}{'POS':<8}{'TAG':<8}{'FREQUENCY':<12}{'DESCRIPTION'}")
for (token, pos, tag), freq in rows:
    print(f'{token:<22}{pos:<8}{tag:<8}{freq:<12}{spacy.explain(tag)}')

Total tokens: 1110
Unique tokens: 408

TOKEN                 POS     TAG     FREQUENCY   DESCRIPTION
,                     PUNCT   ,       76          punctuation mark, comma
.                     PUNCT   .       65          punctuation mark, sentence closer
the                   DET     DT      47          determiner
a                     DET     DT      28          determiner
and                   CCONJ   CC      27          conjunction, coordinating
can                   AUX     MD      23          verb, modal auxiliary
of                    ADP     IN      22          conjunction, subordinating or preposition
subword               PROPN   NNP     16          noun, proper singular
be                    AUX     VB      15          verb, base form
is                    AUX     VBZ     14          verb, 3rd person singular present
as                    ADP     IN      13          conjunction, subordinating or preposition
bpe                   PROPN   NNP     13          noun, proper si

In [27]:
pos_counts = Counter(token.pos_ for token in doc if not token.is_space)

print(f"{'POS':<10}{'FREQUENCY':<12}{'DESCRIPTION'}")
for pos, freq in pos_counts.most_common():
    print(f'{pos:<10}{freq:<12}{spacy.explain(pos)}')

POS       FREQUENCY   DESCRIPTION
NOUN      298         noun
PUNCT     152         punctuation
VERB      130         verb
ADJ       113         adjective
DET       90          determiner
ADP       86          adposition
AUX       71          auxiliary
PROPN     56          proper noun
ADV       41          adverb
CCONJ     32          coordinating conjunction
PRON      15          pronoun
PART      12          particle
SCONJ     11          subordinating conjunction
NUM       2           numeral
X         1           other
